In [ ]:
import cv2
import pandas as pd
import pyttsx3
import numpy as np
from threading import Thread
import queue
import time

# Initialize Text-to-Speech Engine
engine = pyttsx3.init()
engine.setProperty('rate', 180)  # Slightly faster speech for better real-time sync

# Load dataset and normalize RGB values
dataset = pd.read_csv('colors_cleaned_final.csv')
dataset[['R', 'G', 'B']] = dataset[['R', 'G', 'B']] / 255.0  # Normalize RGB

# Thread-safe queue for speech requests
speech_queue = queue.Queue()

# Speak Function (Immediate speech processing)
def speak(text):
    while not speech_queue.empty():  # Clear previous requests to prevent lag
        speech_queue.get_nowait()
    speech_queue.put(text)

# Find closest color with Normalization & Standardization
def detect_color_from_dataset(r, g, b):
    rgb_normalized = np.array([r, g, b]) / 255.0  # Normalize input RGB
    distances = np.linalg.norm(dataset[['R', 'G', 'B']].values - rgb_normalized, axis=1)  # Euclidean distance
    min_index = np.argmin(distances)  # Get the closest color
    return dataset.loc[min_index, "color_name"]

# Optimized Video Capture with Reduced Latency
class VideoStream:
    def __init__(self):
        self.cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
        if not self.cap.isOpened():
            speak("Error: Camera could not be opened.")
            self.running = False
        else:
            self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
            self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
            self.running = True
            self.ret, self.frame = self.cap.read()
            Thread(target=self.update, daemon=True).start()

    def update(self):
        while self.running:
            self.ret, self.frame = self.cap.read()

    def read(self):
        return self.ret, self.frame

    def stop(self):
        self.running = False
        self.cap.release()

# Process Speech Requests Continuously (Low Latency)
def process_speech():
    while True:
        text = speech_queue.get()
        engine.say(text)
        engine.runAndWait()
        speech_queue.task_done()

# Real-Time Color Detection with Synchronization
def start_detection():
    speak("Starting detection and opening the camera.")
    video_stream = VideoStream()

    if not video_stream.running:
        speak("Camera failed to start.")
        return

    prev_color = None
    last_update_time = time.time()

    while True:
        ret, frame = video_stream.read()
        if not ret:
            speak("Failed to capture frame.")
            break

        frame = cv2.flip(frame, 1)
        center_x, center_y = frame.shape[1] // 2, frame.shape[0] // 2
        b, g, r = frame[center_y, center_x]  # Extract center pixel color

        detected_color = detect_color_from_dataset(int(r), int(g), int(b))

        # Instant Update if Color Changes
        if detected_color != prev_color and (time.time() - last_update_time) > 0.5:
            prev_color = detected_color
            last_update_time = time.time()
            speak(f"The detected color is {detected_color}")

        # Display Real-Time Detected Color
        cv2.rectangle(frame, (center_x - 20, center_y - 20), (center_x + 20, center_y + 20), (255, 255, 255), 2)
        cv2.putText(frame, f"Color: {detected_color}", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 3)

        # Show the video feed
        cv2.imshow('Real-Time Color Detection', frame)

        # Press 'q' to quit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            speak("Exiting detection.")
            break

    video_stream.stop()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    speech_thread = Thread(target=process_speech, daemon=True)
    speech_thread.start()
    start_detection()


Exception in thread Thread-5 (process_speech):
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2544.0_x64__qbz5n2kfra8p0\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "e:\AAM Project\.venv\Lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2544.0_x64__qbz5n2kfra8p0\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\ASUS\AppData\Local\Temp\ipykernel_19144\132176011.py", line 63, in process_speech
  File "e:\AAM Project\.venv\Lib\site-packages\pyttsx3\engine.py", line 180, in runAndWait
    raise RuntimeError('run loop already started')
RuntimeError: run loop already started
